# Hand CT 2-Stage nnU-Net Segmentation

This notebook implements a **2-stage segmentation pipeline** for hand CT images using nnU-Net.

**Pipeline overview:**
1. **Data Preparation** — Crop ground-truth ROI regions from the full hand CT and save as separate nnU-Net datasets
2. **Training** — Train one nnU-Net model per ROI group (Thumb, Digits/Metacarpals, Carpals/Wrist)
3. **Inference** — Run Stage 1 (coarse whole-hand) → crop ROIs → run Stage 2 (fine per-ROI) → merge back
4. **Evaluation** — Compare 2-stage Dice scores against the whole-hand baseline

> **Before you start:** Edit the `CONFIGURATION` cell (next section) to point to your data folders.

In [1]:
import torch
print(torch.cuda.is_available())

True


In [2]:
import json
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import SimpleITK as sitk

In [ ]:
# ============================================================
# CONFIGURATION — Edit these paths before running any cells
# ============================================================

# Root folder containing your nnUNet raw data for the ROI experiment.
# Must contain Dataset100_Hand/imagesTr and Dataset100_Hand/labelsTr.
NNUNET_RAW = Path(r"C:\Users\SojungPark\PycharmProjects\nnunet-hand-segmentation\datasets\nnunet_data_250epochs_two-stage\nnUNet_raw")

# The whole-hand dataset (used for Stage 1 coarse segmentation)
SRC_DATASET = NNUNET_RAW / "Dataset100_Hand"

# Which test case to run inference on (file prefix, without _0000.nii)
TEST_CASE = "HAND_008"

# nnUNet_results folder that contains the whole-hand Stage 1 model (Dataset100_HAND).
# This is the results folder from your original baseline experiment — NOT the ROI experiment.
STAGE1_RESULTS_DIR = Path(r"C:\Users\SojungPark\PycharmProjects\nnunet-hand-segmentation\datasets\nnunet_data_250epochs\nnUNet_results")

# Which fold the Stage 1 model was trained on.
# Check your nnUNet_results/Dataset100_HAND/.../fold_* folders to confirm.
# Use "all" if trained with fold_all, or "0","1",... if trained on a specific fold.
STAGE1_FOLD = "all"

# Baseline model's fold_all/validation predictions folder.
# Used in Step 4 to compare 2-stage vs baseline Dice scores.
BASELINE_PRED_DIR = Path(r"C:\Users\SojungPark\PycharmProjects\nnunet-hand-segmentation\datasets\nnunet_data_250epochs\nnUNet_results\Dataset100_HAND\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres\fold_all\validation")

FILE_ENDING = ".nii"

In [11]:
# ROI Groups
ROI_GROUPS = {
    "ThumbROI": {
        "dataset_id": 201,
        "labels": [
            7,   # trapezium
            3,   # scaphoid
            11,  # metacarpal_1
        ],
    },
    "DigitsMetacarpalsROI": {
        "dataset_id": 202,
        "labels": [
            # metacarpals
            11, 12, 13, 14, 15,

            # proximal phalanges 1-5
            16, 17, 18, 19, 20,

            # middle phalanges 2-5
            21, 22, 23, 24,

            # distal phalanges 1-5
            25, 26, 27, 28, 29,
        ],
    },
    "CarpalsWristROI": {
        "dataset_id": 203,
        "labels": [
            # radius / ulna
            1, 2,

            # carpals
            3, 4, 5, 6, 7, 8, 9, 10,
        ],
    },
}

## Step 1: Data Preparation

For each ROI group (ThumbROI, DigitsMetacarpalsROI, CarpalsWristROI):
1. Build a binary mask from the ground-truth label using the ROI's label IDs
2. Compute the smallest bounding box around that mask and expand it by a margin (10 mm)
3. Crop both the CT image and the full multi-class label to that bounding box
4. Save the cropped volumes as a new nnU-Net dataset (Dataset201, 202, 203)

The cell below defines all helper functions. The cell after it runs the pipeline — **only run it once** since it writes files to disk.

In [ ]:
# Helper functions for bounding box, cropping, and dataset creation
# Run this cell once to define all helpers used in Steps 1, 3, and 4.

from typing import Dict, List, Tuple
import json
import numpy as np
import SimpleITK as sitk


# --- Bounding box ---

def get_case_id_from_label_file(label_path: Path) -> str:
    return label_path.name.replace(FILE_ENDING, "")


def get_bbox_from_mask(mask: np.ndarray) -> Tuple[Tuple[int, int], Tuple[int, int], Tuple[int, int]]:
    coords = np.where(mask > 0)
    if len(coords[0]) == 0:
        raise ValueError("Foreground mask is empty.")
    zmin, zmax = coords[0].min(), coords[0].max()
    ymin, ymax = coords[1].min(), coords[1].max()
    xmin, xmax = coords[2].min(), coords[2].max()
    return (zmin, zmax), (ymin, ymax), (xmin, xmax)


def mm_to_voxels(margin_mm: float, spacing_zyx: np.ndarray) -> np.ndarray:
    return np.ceil(margin_mm / spacing_zyx).astype(int)


def expand_bbox(bbox, shape, margin_vox):
    (zmin, zmax), (ymin, ymax), (xmin, xmax) = bbox
    mz, my, mx = margin_vox
    zmin = max(0, zmin - mz)
    zmax = min(shape[0] - 1, zmax + mz)
    ymin = max(0, ymin - my)
    ymax = min(shape[1] - 1, ymax + my)
    xmin = max(0, xmin - mx)
    xmax = min(shape[2] - 1, xmax + mx)
    return (zmin, zmax), (ymin, ymax), (xmin, xmax)


# --- Cropping & ROI masking ---

def crop_array(arr: np.ndarray, bbox):
    (zmin, zmax), (ymin, ymax), (xmin, xmax) = bbox
    return arr[zmin:zmax+1, ymin:ymax+1, xmin:xmax+1]


def make_roi_binary_mask(multiclass_mask: np.ndarray, target_labels: List[int]) -> np.ndarray:
    return np.isin(multiclass_mask, target_labels).astype(np.uint8)


# --- SimpleITK helpers ---

def compute_new_origin(image: sitk.Image, bbox):
    """Convert numpy (z,y,x) bbox corner to SimpleITK physical origin (x,y,z)."""
    (zmin, _), (ymin, _), (xmin, _) = bbox
    return image.TransformIndexToPhysicalPoint((int(xmin), int(ymin), int(zmin)))


def save_cropped_sitk(cropped_array, reference_image, bbox, out_path, is_label=False):
    dtype = np.uint8 if is_label else np.float32
    out_img = sitk.GetImageFromArray(cropped_array.astype(dtype))
    out_img.SetSpacing(reference_image.GetSpacing())
    out_img.SetDirection(reference_image.GetDirection())
    out_img.SetOrigin(compute_new_origin(reference_image, bbox))
    sitk.WriteImage(out_img, str(out_path))


# --- Dataset utilities ---

def make_dataset_json(out_dataset_dir: Path, src_dataset_json_path: Path, dataset_name: str, num_training: int):
    with open(src_dataset_json_path, "r", encoding="utf-8") as f:
        ds = json.load(f)
    ds["name"] = dataset_name
    ds["numTraining"] = num_training
    with open(out_dataset_dir / "dataset.json", "w", encoding="utf-8") as f:
        json.dump(ds, f, indent=2, ensure_ascii=False)


# --- Main pipeline ---

def build_gt_crop_datasets_for_all_rois(src_dataset_dir: Path, dst_root_dir: Path, roi_groups: Dict, margin_mm: float = 15.0):
    src_imagesTr = src_dataset_dir / "imagesTr"
    src_labelsTr = src_dataset_dir / "labelsTr"
    src_dataset_json = src_dataset_dir / "dataset.json"

    label_files = sorted(src_labelsTr.glob(f"*{FILE_ENDING}"))
    if not label_files:
        raise FileNotFoundError(f"No label files found in {src_labelsTr}")

    for roi_name, roi_info in roi_groups.items():
        dataset_id = roi_info["dataset_id"]
        roi_labels = roi_info["labels"]

        dst_dataset_dir = dst_root_dir / f"Dataset{dataset_id:03d}_Hand_{roi_name}"
        dst_imagesTr = dst_dataset_dir / "imagesTr"
        dst_labelsTr = dst_dataset_dir / "labelsTr"
        dst_imagesTr.mkdir(parents=True, exist_ok=True)
        dst_labelsTr.mkdir(parents=True, exist_ok=True)

        saved_cases = 0
        print(f"\nBuilding {dst_dataset_dir.name} ...")

        for label_file in label_files:
            case_id = get_case_id_from_label_file(label_file)
            image_file = src_imagesTr / f"{case_id}_0000{FILE_ENDING}"

            if not image_file.exists():
                print(f"[WARNING] Missing image for {case_id}, skipping.")
                continue

            img_sitk = sitk.ReadImage(str(image_file))
            lbl_sitk = sitk.ReadImage(str(label_file))
            img = sitk.GetArrayFromImage(img_sitk).astype(np.float32)
            lbl = sitk.GetArrayFromImage(lbl_sitk)

            roi_mask = make_roi_binary_mask(lbl, roi_labels)
            if np.sum(roi_mask) == 0:
                print(f"[INFO] {case_id}: ROI not found for {roi_name}, skipped.")
                continue

            bbox = get_bbox_from_mask(roi_mask)
            spacing_zyx = np.array(img_sitk.GetSpacing(), dtype=np.float32)[::-1]
            margin_vox = mm_to_voxels(margin_mm, spacing_zyx)
            bbox = expand_bbox(bbox, img.shape, margin_vox)

            cropped_img = crop_array(img, bbox)
            cropped_lbl = crop_array(lbl, bbox)

            save_cropped_sitk(cropped_img, img_sitk, bbox,
                              dst_imagesTr / f"{case_id}_0000{FILE_ENDING}", is_label=False)
            save_cropped_sitk(cropped_lbl, lbl_sitk, bbox,
                              dst_labelsTr / f"{case_id}{FILE_ENDING}", is_label=True)

            print(f"  {case_id}: original={img.shape}, cropped={cropped_img.shape}, labels={np.unique(cropped_lbl)}")
            saved_cases += 1

        make_dataset_json(dst_dataset_dir, src_dataset_json, dst_dataset_dir.name, saved_cases)
        print(f"[DONE] {dst_dataset_dir.name} — {saved_cases} cases saved.")

In [ ]:
# Run Step 1: build the ROI-cropped datasets
# Only needs to run once — it creates Dataset201, 202, 203 inside nnUNet_raw.
build_gt_crop_datasets_for_all_rois(
    src_dataset_dir=SRC_DATASET,
    dst_root_dir=NNUNET_RAW,
    roi_groups=ROI_GROUPS,
    margin_mm=10.0,
)

## Step 2: Training

Set the nnUNet environment variables, then plan/preprocess and train one model per ROI dataset.

- `nnUNetv2_plan_and_preprocess` analyses each dataset and writes the training plan
- Training runs for 250 epochs on fold 0 (`-f 0`)
- Each ROI dataset (201 Thumb, 202 Digits/Metacarpals, 203 Carpals/Wrist) is trained separately
- Training takes several hours per dataset — run once and save the results

In [ ]:
# set env variables

import os
import subprocess

ROI_DATA_ROOT = NNUNET_RAW.parent

NNUNET_PREPROCESSED = ROI_DATA_ROOT / "nnUNet_preprocessed"
NNUNET_RESULTS      = ROI_DATA_ROOT / "nnUNet_results"

NNUNET_PREPROCESSED.mkdir(parents=True, exist_ok=True)
NNUNET_RESULTS.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"]          = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"]      = str(NNUNET_RESULTS)

print("nnUNet_raw:         ", NNUNET_RAW)
print("nnUNet_preprocessed:", NNUNET_PREPROCESSED)
print("nnUNet_results:     ", NNUNET_RESULTS)


nnUNet_raw:          C:\Users\SojungPark\PycharmProjects\nnunet-hand-segmentation\datasets\nnunet_data_250epochs_left_stage\nnUNet_raw
nnUNet_preprocessed: C:\Users\SojungPark\PycharmProjects\nnunet-hand-segmentation\datasets\nnunet_data_250epochs_left_stage\nnUNet_preprocessed
nnUNet_results:      C:\Users\SojungPark\PycharmProjects\nnunet-hand-segmentation\datasets\nnunet_data_250epochs_left_stage\nnUNet_results


In [16]:
# plan & preprocess

DATASET_IDS = [
    roi_info["dataset_id"] for roi_info in ROI_GROUPS.values()
]  # [201, 202, 203]

for ds_id in DATASET_IDS:
    print(f"\n=== nnUNetv2_plan_and_preprocess  Dataset{ds_id:03d} ===")
    subprocess.run(
        ["nnUNetv2_plan_and_preprocess", "-d", str(ds_id), "--verify_dataset_integrity"],
        check=True,
    )



=== nnUNetv2_plan_and_preprocess  Dataset201 ===

=== nnUNetv2_plan_and_preprocess  Dataset202 ===

=== nnUNetv2_plan_and_preprocess  Dataset203 ===


In [ ]:
# train

import sys

CONFIG  = "3d_fullres"
FOLD    = 0
TRAINER = "nnUNetTrainer_250epochs"

for ds_id in DATASET_IDS:
    print(f"\n=== Training Dataset{ds_id:03d} ===")
    subprocess.run(
        [sys.executable, "-m", "nnunetv2.run.run_training",
         str(ds_id), CONFIG, str(FOLD),
         "-tr", TRAINER],
        check=True,
    )



=== Training Dataset201 ===


## Step 3: Inference

**Stage 1** — Run the trained whole-hand model on the test image to get a coarse segmentation.

**Stage 2** — Use that coarse prediction to locate each ROI, crop the test image to each ROI, run the ROI-specific model, then merge all ROI predictions back into the original full-image space.

To change which case to run inference on, update `TEST_CASE` in the Configuration cell.

In [ ]:
# Stage 1: Run the whole-hand model on the test image to get a coarse segmentation
import sys, shutil

TEST_IMAGE    = ROI_DATA_ROOT / "nnUNet_raw" / "Dataset100_Hand" / "imagesTs" / f"{TEST_CASE}_0000{FILE_ENDING}"
INFER_ROOT    = NNUNET_RAW.parent / "inference_test"
STAGE1_INPUT  = INFER_ROOT / "stage1_input"
STAGE1_OUTPUT = INFER_ROOT / "stage1_output"
STAGE1_INPUT.mkdir(parents=True, exist_ok=True)
STAGE1_OUTPUT.mkdir(parents=True, exist_ok=True)

shutil.copy(TEST_IMAGE, STAGE1_INPUT / TEST_IMAGE.name)

# Use a separate env so nnUNet_results points to the whole-hand model,
# not the ROI experiment folder (Stage 2 models live there instead).
env = os.environ.copy()
env["nnUNet_results"] = str(STAGE1_RESULTS_DIR)

result = subprocess.run([
    sys.executable, "-m", "nnunetv2.inference.predict_from_raw_data",
    "-i", str(STAGE1_INPUT),
    "-o", str(STAGE1_OUTPUT),
    "-d", "100",           # whole-hand dataset (Stage 1)
    "-c", "3d_fullres",
    "-f", STAGE1_FOLD,     # set in Configuration cell
    "-tr", "nnUNetTrainer_250epochs",
    "-npp", "1",
    "-nps", "0",           # synchronous export — avoids spawn+pickle OOM on Windows
], env=env, capture_output=True, text=True)

print("=== STDOUT ===")
print(result.stdout)
print("=== STDERR ===")
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(f"nnUNet Stage 1 inference failed (exit code {result.returncode}). See output above.")

In [ ]:
# Stage 2 prep: use the Stage 1 prediction to crop the test image into per-ROI sub-volumes
CASE_ID = TEST_CASE

img_sitk  = sitk.ReadImage(str(TEST_IMAGE))
pred_sitk = sitk.ReadImage(str(STAGE1_OUTPUT / f"{CASE_ID}{FILE_ENDING}"))

img  = sitk.GetArrayFromImage(img_sitk).astype(np.float32)
pred = sitk.GetArrayFromImage(pred_sitk)

spacing_zyx = np.array(img_sitk.GetSpacing(), dtype=np.float32)[::-1]

ROI_CROP_DIRS = {}

for roi_name, roi_info in ROI_GROUPS.items():
    roi_labels = roi_info["labels"]

    roi_mask = make_roi_binary_mask(pred, roi_labels)  # use Stage 1 prediction, not GT
    if np.sum(roi_mask) == 0:
        print(f"[WARNING] {roi_name}: no foreground found in Stage 1 prediction, skipping.")
        continue

    bbox = get_bbox_from_mask(roi_mask)
    margin_vox = mm_to_voxels(10.0, spacing_zyx)
    bbox = expand_bbox(bbox, img.shape, margin_vox)

    crop_dir = INFER_ROOT / f"stage2_input_{roi_name}"
    crop_dir.mkdir(parents=True, exist_ok=True)

    cropped_img = crop_array(img, bbox)
    save_cropped_sitk(cropped_img, img_sitk, bbox, crop_dir / f"{CASE_ID}_0000{FILE_ENDING}")
    ROI_CROP_DIRS[roi_name] = (crop_dir, bbox)

    print(f"  {roi_name}: cropped shape = {cropped_img.shape}")

In [ ]:
# Stage 2: Run each ROI-specific model on its cropped sub-volume

ROI_PRED_DIRS_TEST = {}

for roi_name, roi_info in ROI_GROUPS.items():
    if roi_name not in ROI_CROP_DIRS:
        continue

    crop_dir, _ = ROI_CROP_DIRS[roi_name]
    out_dir = INFER_ROOT / f"stage2_output_{roi_name}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n=== Inference: {roi_name} ===")
    result = subprocess.run([
        sys.executable, "-m", "nnunetv2.inference.predict_from_raw_data",
        "-i", str(crop_dir),
        "-o", str(out_dir),
        "-d", str(roi_info["dataset_id"]),
        "-c", "3d_fullres",
        "-f", "0",
        "-tr", "nnUNetTrainer_250epochs",
        "-npp", "1",
        "-nps", "0",       # synchronous export — avoids spawn+pickle OOM on Windows
    ], capture_output=True, text=True)

    print("=== STDOUT ===")
    print(result.stdout)
    if result.returncode != 0:
        print("=== STDERR ===")
        print(result.stderr)
        raise RuntimeError(f"Inference failed for {roi_name} (exit code {result.returncode}). See output above.")

    ROI_PRED_DIRS_TEST[roi_name] = out_dir

In [ ]:
# Merge helper

# Which ROI model is authoritative for each label.
# Labels shared across ROIs (e.g. trapezium in both ThumbROI and CarpalsWristROI)
# are resolved here — only the designated ROI's prediction is used.
ROI_PRIMARY_LABELS = {
    "CarpalsWristROI":      [1, 2, 3, 4, 5, 6, 8, 9, 10],   # radius, ulna, carpals (excl. trapezium)
    "ThumbROI":             [7, 11],                          # trapezium + 1st metacarpal
    "DigitsMetacarpalsROI": list(range(12, 30)),              # metacarpals 2-5, all phalanges
}

def merge_roi_predictions(reference_image, roi_pred_sitks, roi_crop_dirs, roi_primary_labels):
    """
    Paste each cropped ROI prediction back into full-image space.
    Each label is written only from its designated primary ROI to avoid conflicts.
    """
    ref_shape = sitk.GetArrayFromImage(reference_image).shape
    merged = np.zeros(ref_shape, dtype=np.uint8)

    for roi_name, pred_sitk in roi_pred_sitks.items():
        primary_labels = roi_primary_labels.get(roi_name, [])
        if not primary_labels:
            continue

        _, bbox = roi_crop_dirs[roi_name]
        (zmin, zmax), (ymin, ymax), (xmin, xmax) = bbox

        pred_arr = sitk.GetArrayFromImage(pred_sitk)

        # --- ADDED: validate prediction shape matches the saved crop bounding box ---
        # If these differ, it means a stale prediction file from a previous failed run
        # is in the output folder. Delete stage2_output_{roi_name} and re-run Stage 2.
        expected_shape = (zmax - zmin + 1, ymax - ymin + 1, xmax - xmin + 1)
        if pred_arr.shape != expected_shape:
            raise ValueError(
                f"[{roi_name}] Prediction shape {pred_arr.shape} does not match "
                f"expected crop shape {expected_shape}.\n"
                f"Stale file from a previous run — delete the folder and re-run Stage 2 inference:\n"
                f"  {roi_crop_dirs[roi_name][0].parent / ('stage2_output_' + roi_name)}"
            )
        # --- END ADDED ---

        for label in primary_labels:
            mask = (pred_arr == label)
            merged[zmin:zmax+1, ymin:ymax+1, xmin:xmax+1][mask] = label

    out = sitk.GetImageFromArray(merged)
    out.CopyInformation(reference_image)
    return out

In [ ]:
# Merge ROI Predictions into full-hand segmentation

roi_pred_sitks = {}
for roi_name, out_dir in ROI_PRED_DIRS_TEST.items():
    pred_file = out_dir / f"{CASE_ID}{FILE_ENDING}"
    if pred_file.exists():
        roi_pred_sitks[roi_name] = sitk.ReadImage(str(pred_file))
    else:
        print(f"[WARNING] Missing prediction for {roi_name}")

merged_sitk = merge_roi_predictions(img_sitk, roi_pred_sitks, ROI_CROP_DIRS, ROI_PRIMARY_LABELS)

MERGED_OUTPUT_DIR = INFER_ROOT
merged_out = MERGED_OUTPUT_DIR / f"{CASE_ID}_merged{FILE_ENDING}"
sitk.WriteImage(merged_sitk, str(merged_out))
print(f"Saved merged prediction → {merged_out}")

In [ ]:
# Evaluation helpers — defines REGION_LABELS and dice_score used by the Evaluate cell below

REGION_LABELS = {
    "radius_ulna": [1, 2],
    "carpals":     [3, 4, 5, 6, 7, 8, 9, 10],
    "metacarpals": [11, 12, 13, 14, 15],
    "phalanges":   list(range(16, 30)),
}

def dice_score(pred: np.ndarray, gt: np.ndarray, label: int) -> float:
    pred_mask = (pred == label)
    gt_mask   = (gt   == label)
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    denom = pred_mask.sum() + gt_mask.sum()
    if denom == 0:
        return float("nan")
    return float(2 * intersection / denom)

In [ ]:
# Step 4: Compare 2-stage vs baseline Dice scores
#
# NOTE: This cell requires the full pipeline to have been run on ALL training cases (HAND_001~007).
# To do that:
#   1. Change TEST_CASE in the Configuration cell to each training case ID
#   2. Re-run Step 3 (Stage 1 → Stage 2 → Merge) for each case
#   3. Then run this cell — it will loop over all cases and average the Dice scores

import pandas as pd

GT_LABELS_DIR     = SRC_DATASET / "labelsTr"
MERGED_OUTPUT_DIR = INFER_ROOT

rows = []
for gt_file in sorted(GT_LABELS_DIR.glob(f"*{FILE_ENDING}")):
    case_id      = gt_file.name.replace(FILE_ENDING, "")
    gt_arr       = sitk.GetArrayFromImage(sitk.ReadImage(str(gt_file)))
    merged_file  = MERGED_OUTPUT_DIR / f"{case_id}_merged{FILE_ENDING}"
    baseline_file = BASELINE_PRED_DIR / f"{case_id}{FILE_ENDING}"

    if not merged_file.exists() or not baseline_file.exists():
        print(f"[SKIP] {case_id}  (merged={merged_file.exists()}, baseline={baseline_file.exists()})")
        continue

    merged_arr   = sitk.GetArrayFromImage(sitk.ReadImage(str(merged_file)))
    baseline_arr = sitk.GetArrayFromImage(sitk.ReadImage(str(baseline_file)))

    for region, labels in REGION_LABELS.items():
        for label in labels:
            rows.append({
                "case":          case_id,
                "region":        region,
                "label":         label,
                "dice_2stage":   dice_score(merged_arr,   gt_arr, label),
                "dice_baseline": dice_score(baseline_arr, gt_arr, label),
            })

if not rows:
    print("No cases to evaluate yet. Run the full pipeline on training cases first.")
else:
    df = pd.DataFrame(rows)
    summary = (df.groupby("region")[["dice_2stage", "dice_baseline"]]
                 .mean()
                 .round(3))
    print(summary)